# AirShift — Labeling

This notebook applies the deterioration event definition established in the previous stage to create the target label for machine learning.

The labeling process focuses on:

* Applying the selected PM2.5 deterioration threshold
* Using the selected future prediction horizon
* Creating a binary target variable
* Handling observations with insufficient information for labeling
* Validating the resulting class distribution
* Saving the labeled dataset for model development

The target label represents whether a future air quality deterioration event occurs according to the predefined event definition.

No model training is performed in this notebook.


## 1. Load the Feature-Engineered Dataset

The feature-engineered dataset created in the previous stage is loaded as the starting point for the labeling process.

The dataset contains the temporal and historical air quality features required to define the deterioration target.

The original feature-engineered dataset is not modified or overwritten.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
DATA_DIR = Path("../data/processed")
OUTPUT_DIR = Path("../data/processed")

feature_file = DATA_DIR / "airshift_feature_engineered.csv"

df = pd.read_csv(
    feature_file,
    parse_dates=["datetime"]
)

print("Dataset shape:", df.shape)
print("Number of stations:", df["station"].nunique())
print("Date range:", df["datetime"].min(), "to", df["datetime"].max())

Dataset shape: (420768, 99)
Number of stations: 12
Date range: 2013-03-01 00:00:00 to 2017-02-28 23:00:00


## 2. Define the Labeling Configuration

The labeling configuration is defined according to the deterioration event definition established in the previous notebook.

The target is based on PM2.5, with a **30% deterioration threshold** and a **6-hour future horizon**.

These parameters will be used to create the binary target label in the following steps.


In [ ]:
TARGET_POLLUTANT = "PM2.5"
FINAL_THRESHOLD = 0.30
FINAL_HORIZON_HOURS = 6

print("Labeling configuration:")
print(f"Target pollutant: {TARGET_POLLUTANT}")
print(f"Deterioration threshold: {FINAL_THRESHOLD * 100:.0f}%")
print(f"Future horizon: {FINAL_HORIZON_HOURS} hours")

Labeling configuration:
Target pollutant: PM2.5
Deterioration threshold: 30%
Future horizon: 6 hours


In [4]:
# Sort the dataset by monitoring station and datetime to preserve the
# chronological order required for accurate time-based labeling.
df = df.sort_values(
    ["station", "datetime"]
).reset_index(drop=True)

print("Data sorted successfully.")

Data sorted successfully.


## 3. Check PM2.5 Availability

Before creating the target label, the availability of the current PM2.5 values is checked.

A valid current PM2.5 value is required to calculate the relative change and determine whether a deterioration event occurs.

This step only evaluates missing values and does not remove or modify any observations.


In [5]:
# Check the number and percentage of missing current PM2.5 values.
pm25_missing = df["PM2.5"].isna().sum()

pm25_missing_percentage = (
    pm25_missing / len(df)
) * 100

print("Missing current PM2.5 values:", pm25_missing)
print(f"Missing percentage: {pm25_missing_percentage:.2f}%")

Missing current PM2.5 values: 1877
Missing percentage: 0.45%


### PM2.5 Availability Findings

The dataset contains **1,877 missing current PM2.5 values**, representing **0.45%** of all observations.

These observations will not be assigned a deterioration label because a valid current PM2.5 value is required to calculate the relative change.

The observations will be handled during the labeling process without creating artificial target values.


## 4. Calculate Future PM2.5 Values

To identify whether air quality deteriorates within the next 6 hours, the future PM2.5 concentration is examined across the six hourly observations following each prediction time.

For each observation, the maximum PM2.5 value within the next 6 hours is calculated separately for each monitoring station.

This approach matches the AirShift objective of detecting deterioration that may occur at any point during the future warning horizon rather than only at exactly 6 hours later.


In [6]:
# Calculate the maximum PM2.5 concentration observed during the
# following 6 hours for each monitoring station.

future_pm25_values = []

for hour in range(1, FINAL_HORIZON_HOURS + 1):
    future_pm25_values.append(
        df.groupby("station")[TARGET_POLLUTANT].shift(-hour)
    )

df["PM2.5_future_max_6h"] = pd.concat(
    future_pm25_values,
    axis=1
).max(axis=1)

print("Future maximum PM2.5 values calculated successfully.")

df[
    [
        "station",
        "datetime",
        "PM2.5",
        "PM2.5_future_max_6h"
    ]
].head(10)

Future maximum PM2.5 values calculated successfully.


,station,datetime,PM2.5,PM2.5_future_max_6h
0,Aotizhongxin,2013-03-01 00:00:00,4.0,8.0
1,Aotizhongxin,2013-03-01 01:00:00,8.0,7.0
2,Aotizhongxin,2013-03-01 02:00:00,7.0,6.0
3,Aotizhongxin,2013-03-01 03:00:00,6.0,5.0
4,Aotizhongxin,2013-03-01 04:00:00,3.0,5.0
5,Aotizhongxin,2013-03-01 05:00:00,5.0,3.0
6,Aotizhongxin,2013-03-01 06:00:00,3.0,3.0
7,Aotizhongxin,2013-03-01 07:00:00,3.0,3.0
8,Aotizhongxin,2013-03-01 08:00:00,3.0,6.0
9,Aotizhongxin,2013-03-01 09:00:00,3.0,8.0


### Future PM2.5 Findings

The maximum PM2.5 concentration within the following 6 hours was calculated separately for each monitoring station.

Using the maximum future value ensures that deterioration occurring at any point within the 6-hour prediction horizon can be captured.

These future values will be compared with the current PM2.5 concentration in the next step to calculate the relative deterioration.


## 5. Calculate Future PM2.5 Change

The relative change in PM2.5 is calculated by comparing the current PM2.5 concentration with the maximum concentration observed during the following 6 hours.

A positive value indicates that PM2.5 increased during the future horizon, while a negative value indicates that the future maximum remained below the current concentration.

The relative change will be used in the next step to determine whether the predefined 30% deterioration threshold is reached.


In [7]:
df["PM2.5_future_change_6h"] = np.where(
    df[TARGET_POLLUTANT] > 0,
    (
        df["PM2.5_future_max_6h"] - df[TARGET_POLLUTANT]
    ) / df[TARGET_POLLUTANT],
    np.nan
)

print("Future PM2.5 change calculated successfully.")

df[
    [
        "station",
        "datetime",
        "PM2.5",
        "PM2.5_future_max_6h",
        "PM2.5_future_change_6h"
    ]
].head(10)

Future PM2.5 change calculated successfully.


,station,datetime,PM2.5,PM2.5_future_max_6h,PM2.5_future_change_6h
0,Aotizhongxin,2013-03-01 00:00:00,4.0,8.0,1.000000
1,Aotizhongxin,2013-03-01 01:00:00,8.0,7.0,-0.125000
2,Aotizhongxin,2013-03-01 02:00:00,7.0,6.0,-0.142857
3,Aotizhongxin,2013-03-01 03:00:00,6.0,5.0,-0.166667
4,Aotizhongxin,2013-03-01 04:00:00,3.0,5.0,0.666667
5,Aotizhongxin,2013-03-01 05:00:00,5.0,3.0,-0.400000
6,Aotizhongxin,2013-03-01 06:00:00,3.0,3.0,0.000000
7,Aotizhongxin,2013-03-01 07:00:00,3.0,3.0,0.000000
8,Aotizhongxin,2013-03-01 08:00:00,3.0,6.0,1.000000
9,Aotizhongxin,2013-03-01 09:00:00,3.0,8.0,1.666667


### Future PM2.5 Change Findings

The relative change in PM2.5 was calculated by comparing the current concentration with the maximum concentration observed during the following 6 hours.

Positive values indicate an increase in PM2.5 during the future horizon, while negative values indicate that the future maximum remained below the current concentration.

For example, an increase from 3.0 to 6.0 results in a relative change of **1.0**, corresponding to a **100% increase**.

The resulting relative change will be used to determine whether the **30% deterioration threshold** is reached when creating the target label.
